"""
Cálculo de curvatura (estándar, perfil, plano) replicando exactamente
la función `Curvature` de ArcGIS (algoritmo Zevenbergen & Thorne 1987).

Convenciones (idénticas a ArcGIS):
- Curvatura estándar negativa = cóncava ; positiva = convexa
- Curvatura de perfil negativa = convexa hacia arriba (desacelera flujo)
- Curvatura de plano  positiva = convexa lateralmente (diverge flujo)
- Unidades = 1/100 de la unidad z

Referencia:
    Zevenbergen, L.W. & Thorne, C.R. (1987). Quantitative analysis of land
    surface topography. Earth Surface Processes and Landforms, 12, 47–56.
"""

In [6]:


from pathlib import Path
from typing import Union, Dict
import numpy as np
import rasterio


def _coef_zevenbergen_thorne(dem: np.ndarray, L: float):
    """
    Calcula los coeficientes D, E, F, G, H de la superficie cuadrática de
    Zevenbergen & Thorne (1987) para cada pixel interno.
    Las celdas del borde quedan implícitamente en NaN al recortar.
    """
    Z1 = dem[:-2, :-2]
    Z2 = dem[:-2, 1:-1]
    Z3 = dem[:-2, 2:]
    Z4 = dem[1:-1, :-2]
    Z5 = dem[1:-1, 1:-1]
    Z6 = dem[1:-1, 2:]
    Z7 = dem[2:,   :-2]
    Z8 = dem[2:,   1:-1]
    Z9 = dem[2:,   2:]

    D = ((Z4 + Z6) / 2.0 - Z5) / (L ** 2)
    E = ((Z2 + Z8) / 2.0 - Z5) / (L ** 2)
    F = (-Z1 + Z3 + Z7 - Z9) / (4.0 * L ** 2)
    G = (-Z4 + Z6) / (2.0 * L)
    H = (Z2 - Z8) / (2.0 * L)

    return D, E, F, G, H


def curvaturas_arcgis(dem: np.ndarray, cell_size: float) -> Dict[str, np.ndarray]:
    """
    Calcula las tres curvaturas (estándar, perfil, plano) replicando
    la convención de ArcGIS.

    Parámetros
    ----------
    dem : np.ndarray (2D)
        Modelo de elevación digital. Las celdas con NaN se propagan.
    cell_size : float
        Tamaño de celda en metros (debe coincidir con la unidad z).

    Retorna
    -------
    dict con tres arrays (mismo shape que dem):
        "estandar", "perfil", "plano".
    """
    L = float(cell_size)
    D, E, F, G, H = _coef_zevenbergen_thorne(dem, L)

    estandar = np.full_like(dem, np.nan, dtype="float64")
    perfil   = np.full_like(dem, np.nan, dtype="float64")
    plano    = np.full_like(dem, np.nan, dtype="float64")

    # Curvatura estándar
    estandar[1:-1, 1:-1] = -2.0 * (D + E) * 100.0

    # Curvaturas direccionales: cuidado con G²+H² = 0 (zona plana)
    GH2 = G ** 2 + H ** 2
    mask = GH2 > 1e-12

    perfil_val = np.full_like(D, np.nan)
    plano_val  = np.full_like(D, np.nan)

    perfil_val[mask] = (
        2.0 * (D[mask] * G[mask] ** 2
               + E[mask] * H[mask] ** 2
               + F[mask] * G[mask] * H[mask])
        / GH2[mask]
    ) * 100.0

    plano_val[mask] = (
        -2.0 * (D[mask] * H[mask] ** 2
                + E[mask] * G[mask] ** 2
                - F[mask] * G[mask] * H[mask])
        / GH2[mask]
    ) * 100.0

    # En zonas planas, perfil y plano = 0 por convención ArcGIS
    perfil_val[~mask] = 0.0
    plano_val[~mask]  = 0.0

    perfil[1:-1, 1:-1] = perfil_val
    plano[1:-1, 1:-1]  = plano_val

    return {"estandar": estandar, "perfil": perfil, "plano": plano}


def procesar_dem(
    dem_path: Union[str, Path],
    out_dir: Union[str, Path],
    nodata_out: float = -9999.0,
) -> Dict[str, Path]:
    """
    Lee un DEM, calcula las tres curvaturas y las guarda como GeoTIFF.

    Genera tres archivos:
        curvatura_estandar.tif
        curvatura_perfil.tif
        curvatura_plano.tif
    """
    dem_path = Path(dem_path)
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    with rasterio.open(dem_path) as src:
        dem_arr = src.read(1).astype("float64")
        profile = src.profile
        nodata_in = src.nodata
        cell_size = abs(src.transform.a)
        res_y = abs(src.transform.e)

    if nodata_in is not None:
        dem_arr[dem_arr == nodata_in] = np.nan

    if not np.isclose(cell_size, res_y, rtol=1e-3):
        print(
            "[Aviso] Resolución x ({}) y resolución y ({}) "
            "no son iguales. Z&T asume celdas cuadradas; "
            "el resultado puede ser aproximado.".format(cell_size, res_y)
        )

    print("DEM cargado: shape={}, cell_size={} m".format(dem_arr.shape, cell_size))

    curvas = curvaturas_arcgis(dem_arr, cell_size)

    profile.update(
        dtype="float32",
        count=1,
        nodata=nodata_out,
        compress="deflate",
        predictor=2,          # ← mejora compresión en datos continuos (~20-30%)
        tiled=True,           # ← escritura por tiles, más robusta para rasters grandes
        blockxsize=512,
        blockysize=512,
        zlevel=6,
    )

    salidas = {}
    for tipo, arr in curvas.items():
        out_path = out_dir / "curvatura_{}.tif".format(tipo)
        arr_out = arr.astype("float32")
        arr_out[np.isnan(arr_out)] = nodata_out
        with rasterio.open(out_path, "w", **profile) as dst:
            dst.write(arr_out, 1)
        salidas[tipo] = out_path
        print("  → {}: min={:.3f}, max={:.3f}, media={:.4f}".format(
            out_path.name,
            np.nanmin(arr),
            np.nanmax(arr),
            np.nanmean(arr),
        ))

    return salidas




In [ ]:
DEM = "/home/oisanchezp/Thesis/data/metadata/dem_AMVA_5m_3116.tif"
OUT = "/home/oisanchezp/Thesis/src/04_Chapter/Curvatura"
procesar_dem(DEM, OUT)

# Pequeña adaptación: calcular las tres pero guardar solo estándar
from pathlib import Path
import rasterio
import numpy as np



DEM cargado: shape=(11805, 11157), cell_size=5.000261093351383 m
  → curvatura_estandar.tif: min=-350.935, max=498.269, media=0.0001
  → curvatura_perfil.tif: min=-247.126, max=250.023, media=0.0153


RasterioIOError: Read or write failed. TIFFAppendToStrip:Write error at scanline 4096

##  Validación del resultado

In [8]:
import rasterio
import numpy as np

with rasterio.open("/home/oisanchezp/Thesis/src/04_Chapter/Curvatura/curvatura_estandar.tif") as src:
    c = src.read(1)
    c = c[c != src.nodata]

print(f"Min:      {c.min():.3f}")
print(f"Max:      {c.max():.3f}")
print(f"Media:    {c.mean():.4f}")  # debería estar muy cerca de 0
print(f"Mediana:  {np.median(c):.4f}")
print(f"P1, P99:  {np.percentile(c, 1):.3f}, {np.percentile(c, 99):.3f}")

Min:      -350.935
Max:      498.269
Media:    0.0001
Mediana:  0.0195
P1, P99:  -9.077, 7.389


In [9]:
import geopandas as gpd
from rasterstats import zonal_stats
import rasterio

su = gpd.read_file("/home/oisanchezp/Thesis/src/04_Chapter/Slope_detallado/slope_units_final_mm.gpkg")
ras = "/home/oisanchezp/Thesis/src/04_Chapter/Curvatura/curvatura_estandar.tif"

with rasterio.open(ras) as src:
    su = su.to_crs(src.crs)

stats = zonal_stats(su, ras, stats=["mean", "std"], nodata=-9999)
su["curva_std_mean"] = [s["mean"] for s in stats]
su["curva_std_std"]  = [s["std"]  for s in stats]

su.to_file("/home/oisanchezp/Thesis/src/04_Chapter/Slope_detallado/slope_units_final_mm.gpkg", driver="GPKG")